# USDJPY All-In-One Colab Trainer
Upload your usdjpy_m1.csv to the Colab workspace, then run all cells. This notebook will recreate the Python files and run the pipeline.

In [ ]:
!pip install lightgbm shap pandas numpy scikit-learn numba ta


### Create synthetic_microstructure.py

In [ ]:
%%writefile synthetic_microstructure.py
#!/usr/bin/env python3
"""
synthetic_microstructure.py
---------------------------
Derive tick-free microstructure proxies directly from Dukascopy M1 OHLCV.

Why this exists
===============
Your Dukascopy bulk loader gives you M1 OHLC + tick_volume but NOT real ticks,
so real `microstructure_features.py` (tick_imbalance, OFI, spread, volume
profile from ticks) can only run on the small FTMO/MT5 tick window. This
leaves years of training data without microstructure signal.

This module replaces the missing tick information with *academically
validated* OHLCV-derived proxies:

    1. Corwin–Schultz (2012) & Abdi–Ranaldo (2017) spread estimators
       -> proxies for `spread_mean` / `spread_std`.

    2. Tick-rule signed volume via Lee–Ready: for each M1 bar we sign the
       net tick_volume by sign(close_i - close_{i-1}); then we build
       buy_vol / sell_vol and an OFI proxy normalized by rolling volume.

    3. Kyle-lambda proxy (|return| / volume) -> liquidity / price impact.
       Amihud illiquidity (rolling) for regime filtering.

    4. Tick Imbalance Bars (TIB) & Dollar Imbalance Bars (DIB) of
       López de Prado: event-driven resampling that yields bars with
       homogeneous information content — crucial when you don't have
       real ticks. Implemented using the signed tick_volume above.

    5. Volume-profile-from-bars: approximate VPOC / value area inside a
       rolling window by spreading each bar's volume across its [low, high]
       range. Gives you `vprof_poc_dist`, `vprof_in_value_area`, HVN/LVN
       without ticks.

    6. Jump / microstructure noise flags: Corsi et al. style jump test
       using (close-open) vs rolling vol for event flags.

All features use ONLY columns you already have from Dukascopy:
    time, open, high, low, close, tick_volume

Output columns are deliberately named the same way as
`microstructure_features.py` so that `train_ensemble_gpu.py --micro`
can consume the enriched file with ZERO code changes:

    tick_imbalance, bid_ask_vol_imbalance, spread_mean, spread_std,
    ofi_window, of_pressure_flag,
    vprof_poc_dist, vprof_in_value_area, vprof_hvn_flag, vprof_lvn_flag

Plus extra columns that are genuinely new signal:
    kyle_lambda, amihud_illiq, cs_spread, ar_spread,
    jump_flag, signed_vol_z, vol_regime

Usage
-----
    python train_pipeline/synthetic_microstructure.py \
        --m1  train_pipeline/data/xauusd_m1.csv \
        --out train_pipeline/data/xauusd_m1_synmicro.csv \
        --vp-window 240 \
        --bin-size 0.10

Then retrain as usual with `--micro`:
    python train_pipeline/train_ensemble_gpu.py \
        --data train_pipeline/data/xauusd_m1_synmicro.csv \
        --expanded-features --micro ...

References
----------
  Corwin, S. & Schultz, P. (2012) "A Simple Way to Estimate Bid-Ask
    Spreads from Daily High and Low Prices." J. Finance.
  Abdi, F. & Ranaldo, A. (2017) "A Simple Estimation of Bid-Ask Spreads
    from Daily Close, High, and Low Prices." RFS.
  Lee, C. & Ready, M. (1991) "Inferring Trade Direction from Intraday
    Data." J. Finance.
  López de Prado, M. (2018) "Advances in Financial Machine Learning",
    Ch. 2 (Information-Driven Bars), Ch. 3 (Triple-Barrier & Meta-
    Labeling).
  Amihud, Y. (2002) "Illiquidity and stock returns."
"""

from __future__ import annotations

import argparse
import logging
import sys
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("SynMicro")

TICK_IMBALANCE_STRONG = 0.30
VALUE_AREA_FRACTION = 0.70
HVN_PERCENTILE = 80
LVN_PERCENTILE = 20


# ---------------------------------------------------------------------------
# 1. Spread estimators (OHLC only)
# ---------------------------------------------------------------------------

def corwin_schultz_spread(df: pd.DataFrame) -> pd.Series:
    """Corwin–Schultz 2-period high–low spread estimator.

    S = 2 * (exp(alpha) - 1) / (1 + exp(alpha))
    alpha = (sqrt(2*beta) - sqrt(beta)) / (3 - 2*sqrt(2)) - sqrt(gamma / (3 - 2*sqrt(2)))
      beta  = E[(ln H_t/L_t)^2 + (ln H_{t-1}/L_{t-1})^2]
      gamma = (ln H_{t,t-1} / L_{t,t-1})^2  (using max/min over 2 bars)

    Negative spreads are clipped to 0 per standard treatment.
    """
    h = df["high"].astype("float64")
    l = df["low"].astype("float64")
    # Adjust for overnight gaps: if today's low > yesterday's high, shift low up.
    h1, l1 = h.shift(1), l.shift(1)
    # 2-bar high / low
    h2 = pd.concat([h, h1], axis=1).max(axis=1)
    l2 = pd.concat([l, l1], axis=1).min(axis=1)

    beta = (np.log(h / l) ** 2) + (np.log(h1 / l1) ** 2)
    gamma = np.log(h2 / l2) ** 2

    k1 = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k1 - np.sqrt(gamma / k1)
    S = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    return S.clip(lower=0).astype("float32")


def abdi_ranaldo_spread(df: pd.DataFrame) -> pd.Series:
    """Abdi–Ranaldo 2-period spread estimator (AR 2017).

    S^2 = 4 * E[(c_t - eta_t) * (c_t - eta_{t+1})]
    where eta_t = (h_t + l_t) / 2.
    """
    c = np.log(df["close"].astype("float64"))
    h = np.log(df["high"].astype("float64"))
    l = np.log(df["low"].astype("float64"))
    eta = (h + l) / 2.0
    # (c_t - eta_t) * (c_t - eta_{t+1})
    term = (c - eta) * (c - eta.shift(-1))
    S2 = 4.0 * term
    S = np.sqrt(S2.clip(lower=0))
    return S.astype("float32")


# ---------------------------------------------------------------------------
# 2. Signed volume via tick rule (Lee–Ready bar-level)
# ---------------------------------------------------------------------------

def signed_tick_volume(df: pd.DataFrame) -> pd.DataFrame:
    """Apply Lee–Ready style tick rule at bar level.

    Sign convention:
        +1 if close > prev_close
        -1 if close < prev_close
         0 if unchanged -> carry last sign forward (reverse tick test)

    Buy_vol / sell_vol split tick_volume proportional to intrabar range
    (open->close fraction) to add a 2nd-order refinement a la Easley/O'Hara.
    """
    out = pd.DataFrame(index=df.index)
    dc = df["close"].diff()
    s = np.sign(dc)
    # Reverse tick for zeros: forward-fill last non-zero sign
    s = s.replace(0, np.nan).ffill().fillna(0).astype("int8")

    # Refined split using body/range (Nyquist-style fraction)
    rng = (df["high"] - df["low"]).replace(0, np.nan)
    body_up = (df["close"] - df["open"]).clip(lower=0) / rng
    body_up = body_up.fillna(0.5).clip(0, 1)
    # buy fraction: when close > open dominate, else reduce
    frac_buy = np.where(s > 0, 0.5 + 0.5 * body_up,
                np.where(s < 0, 0.5 - 0.5 * body_up, 0.5))
    vol = df["tick_volume"].astype("float32")
    out["buy_vol"] = (vol * frac_buy).astype("float32")
    out["sell_vol"] = (vol * (1 - frac_buy)).astype("float32")
    out["signed_vol"] = (out["buy_vol"] - out["sell_vol"]).astype("float32")
    out["tick_sign"] = s
    return out


def rolling_ofi_proxy(signed_vol: pd.Series, window: int = 20) -> pd.Series:
    """OFI proxy = rolling sum(signed_vol) / rolling sum(|signed_vol|)."""
    num = signed_vol.rolling(window, min_periods=1).sum()
    den = signed_vol.abs().rolling(window, min_periods=1).sum().replace(0, np.nan)
    return (num / den).fillna(0).astype("float32")


# ---------------------------------------------------------------------------
# 3. Liquidity / price-impact proxies
# ---------------------------------------------------------------------------

def liquidity_proxies(df: pd.DataFrame, window: int = 60) -> pd.DataFrame:
    """Kyle's lambda (|r|/V) and Amihud illiquidity as bar-level proxies."""
    r = df["close"].pct_change().abs()
    v = df["tick_volume"].astype("float64").replace(0, np.nan)
    lam = (r / v).replace([np.inf, -np.inf], np.nan)
    out = pd.DataFrame(index=df.index)
    out["kyle_lambda"] = lam.rolling(window, min_periods=5).mean().astype("float32")
    out["amihud_illiq"] = (r / (v * df["close"])).rolling(window, min_periods=5).mean().astype("float32")
    return out


# ---------------------------------------------------------------------------
# 4. Volume profile built from OHLC bars (triangular distribution)
# ---------------------------------------------------------------------------

def _distribute_bar_volume(low: float, high: float, close: float, vol: float, bin_size: float):
    """Spread `vol` across price bins in [low, high] with a triangular weight
    peaked at the close. Returns {bin_center: volume_contribution}.
    """
    if vol <= 0 or high <= low or np.isnan(low) or np.isnan(high):
        return {}
    lo_b = round(low / bin_size) * bin_size
    hi_b = round(high / bin_size) * bin_size
    n = int(round((hi_b - lo_b) / bin_size)) + 1
    if n <= 1:
        return {lo_b: float(vol)}
    # Triangular weights peaked at the bin containing `close`
    centers = np.array([lo_b + i * bin_size for i in range(n)])
    peak = round(close / bin_size) * bin_size
    w = 1.0 - np.abs(centers - peak) / max((hi_b - lo_b) / 2, bin_size)
    w = np.clip(w, 0.05, 1.0)
    w = w / w.sum()
    return {float(centers[i]): float(vol * w[i]) for i in range(n)}


def rolling_volume_profile_from_bars(
    df: pd.DataFrame, vp_window: int = 240, bin_size: float = 0.10
) -> pd.DataFrame:
    """Sliding-window volume profile computed only from bar volume + OHL.

    For each bar t, we build the profile over the last `vp_window` bars by
    summing the triangular distribution of each bar's volume across its
    [low, high] range. This reproduces the *statistical* shape of a tick-
    derived volume profile well enough for HVN/LVN/value-area features.
    """
    # Pre-compute bin dicts per bar (vectorized loop — cheap)
    logger.info(f"  Pre-computing per-bar bin maps ({len(df):,} bars)")
    per_bar = [
        _distribute_bar_volume(
            float(df["low"].iat[i]),
            float(df["high"].iat[i]),
            float(df["close"].iat[i]),
            float(df["tick_volume"].iat[i]),
            bin_size,
        )
        for i in range(len(df))
    ]

    # Rolling aggregation
    window_q: deque = deque()
    rolling_vol: defaultdict = defaultdict(float)

    # ATR for normalization of POC distance
    atr_col = df["ATR"] if "ATR" in df.columns else None

    results = []
    for i in range(len(df)):
        cur = per_bar[i]
        window_q.append((i, cur))
        for b, c in cur.items():
            rolling_vol[b] += c

        while len(window_q) > vp_window:
            _, old = window_q.popleft()
            for b, c in old.items():
                rolling_vol[b] -= c
                if rolling_vol[b] <= 1e-9:
                    del rolling_vol[b]

        if not rolling_vol:
            results.append((np.nan,) * 7)
            continue

        total = sum(rolling_vol.values())
        poc_bin = max(rolling_vol, key=rolling_vol.get)
        poc_vol = rolling_vol[poc_bin]

        sorted_bins = sorted(rolling_vol.keys())
        poc_i = sorted_bins.index(poc_bin)

        accumulated = poc_vol
        lo_i, hi_i = poc_i, poc_i
        target = total * VALUE_AREA_FRACTION
        while accumulated < target:
            can_lo = lo_i > 0
            can_hi = hi_i < len(sorted_bins) - 1
            if not can_lo and not can_hi:
                break
            add_lo = rolling_vol.get(sorted_bins[lo_i - 1], 0) if can_lo else -1
            add_hi = rolling_vol.get(sorted_bins[hi_i + 1], 0) if can_hi else -1
            if add_hi >= add_lo:
                hi_i += 1
                accumulated += add_hi
            else:
                lo_i -= 1
                accumulated += add_lo
        va_high = sorted_bins[hi_i]
        va_low = sorted_bins[lo_i]

        cur_close = df["close"].iat[i]
        cur_bin = round(cur_close / bin_size) * bin_size
        cur_bin_vol = rolling_vol.get(cur_bin, 0)

        non_zero = [v for v in rolling_vol.values() if v > 0]
        if non_zero:
            hvn_thresh = np.percentile(non_zero, HVN_PERCENTILE)
            lvn_thresh = np.percentile(non_zero, LVN_PERCENTILE)
            hvn = int(cur_bin_vol >= hvn_thresh)
            lvn = int(0 < cur_bin_vol <= lvn_thresh)
        else:
            hvn = lvn = 0

        atr_val = atr_col.iat[i] if atr_col is not None else np.nan
        if np.isnan(atr_val) or atr_val <= 0:
            # fallback: use recent range std
            atr_val = max((df["high"].iat[i] - df["low"].iat[i]), 0.1)
        poc_dist = (cur_close - poc_bin) / atr_val

        results.append((poc_bin, poc_dist, va_high, va_low,
                        int(va_low <= cur_close <= va_high), hvn, lvn))

    cols = ["vprof_poc_price", "vprof_poc_dist", "vprof_va_high",
            "vprof_va_low", "vprof_in_value_area",
            "vprof_hvn_flag", "vprof_lvn_flag"]
    vp_df = pd.DataFrame(results, columns=cols, index=df.index)
    vp_df["vprof_poc_price"] = vp_df["vprof_poc_price"].astype("float32")
    vp_df["vprof_poc_dist"] = vp_df["vprof_poc_dist"].astype("float32")
    vp_df["vprof_va_high"] = vp_df["vprof_va_high"].astype("float32")
    vp_df["vprof_va_low"] = vp_df["vprof_va_low"].astype("float32")
    vp_df["vprof_in_value_area"] = vp_df["vprof_in_value_area"].fillna(0).astype("int8")
    vp_df["vprof_hvn_flag"] = vp_df["vprof_hvn_flag"].fillna(0).astype("int8")
    vp_df["vprof_lvn_flag"] = vp_df["vprof_lvn_flag"].fillna(0).astype("int8")
    return vp_df


# ---------------------------------------------------------------------------
# 5. Jump / regime flags
# ---------------------------------------------------------------------------

def jump_and_regime(df: pd.DataFrame, window: int = 60) -> pd.DataFrame:
    """Jump flag using Lee–Mykland-style standardized return test, and a
    volatility regime label (low/mid/high) from rolling realized vol."""
    r = df["close"].pct_change()
    rv = r.rolling(window, min_periods=10).std()
    z = (r / rv).replace([np.inf, -np.inf], np.nan).fillna(0)
    jump = (z.abs() > 4.0).astype("int8")  # 4-sigma jumps
    # vol regime: 0 = low, 1 = mid, 2 = high
    q1 = rv.rolling(window * 10, min_periods=window).quantile(0.33)
    q2 = rv.rolling(window * 10, min_periods=window).quantile(0.66)
    regime = np.where(rv <= q1, 0, np.where(rv >= q2, 2, 1)).astype("int8")
    return pd.DataFrame({
        "jump_flag": jump,
        "signed_vol_z": z.astype("float32"),
        "vol_regime": regime,
    }, index=df.index)


# ---------------------------------------------------------------------------
# 6. Main assembly — writes columns matching microstructure_features.py
# ---------------------------------------------------------------------------

def build_synthetic_microstructure(
    m1_path: str,
    out_path: str,
    vp_window: int = 240,
    bin_size: float = 0.10,
    ofi_window: int = 20,
) -> pd.DataFrame:
    logger.info(f"Loading M1 bars: {m1_path}")
    df = pd.read_csv(m1_path)
    df.columns = [c.lower().strip() for c in df.columns]
    for c in ["open", "high", "low", "close", "tick_volume"]:
        if c not in df.columns:
            logger.error(f"Missing required column {c}")
            sys.exit(1)
    df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
    df = df.sort_values("time").reset_index(drop=True)

    logger.info("Computing ATR for normalization")
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift()).abs()
    lc = (df["low"] - df["close"].shift()).abs()
    df["ATR"] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(14).mean()

    logger.info("1) Spread estimators (CS + AR)")
    cs = corwin_schultz_spread(df)
    ar = abdi_ranaldo_spread(df)
    df["cs_spread"] = cs
    df["ar_spread"] = ar
    # Expose as the column name downstream expects
    df["spread_mean"] = cs.rolling(20, min_periods=1).mean().astype("float32")
    df["spread_std"] = cs.rolling(20, min_periods=1).std().fillna(0).astype("float32")

    logger.info("2) Signed-volume tick rule + OFI proxy")
    sv = signed_tick_volume(df)
    df["buy_vol"] = sv["buy_vol"]
    df["sell_vol"] = sv["sell_vol"]
    df["signed_vol"] = sv["signed_vol"]
    df["tick_sign"] = sv["tick_sign"]
    # tick_imbalance on a rolling window (keeps name for downstream)
    num = sv["tick_sign"].rolling(ofi_window, min_periods=1).sum()
    df["tick_imbalance"] = (num / ofi_window).astype("float32")
    df["bid_ask_vol_imbalance"] = (
        (sv["buy_vol"] - sv["sell_vol"])
        / (sv["buy_vol"] + sv["sell_vol"] + 1e-9)
    ).astype("float32")
    df["ofi_window"] = rolling_ofi_proxy(sv["signed_vol"], ofi_window)
    df["of_pressure_flag"] = np.select(
        [df["tick_imbalance"] > TICK_IMBALANCE_STRONG,
         df["tick_imbalance"] < -TICK_IMBALANCE_STRONG],
        [1, -1], default=0).astype("int8")

    logger.info("3) Liquidity proxies (Kyle lambda, Amihud)")
    liq = liquidity_proxies(df)
    df["kyle_lambda"] = liq["kyle_lambda"]
    df["amihud_illiq"] = liq["amihud_illiq"]

    logger.info(f"4) Volume profile from bars (window={vp_window}, bin={bin_size})")
    vp = rolling_volume_profile_from_bars(df, vp_window=vp_window, bin_size=bin_size)
    for c in vp.columns:
        df[c] = vp[c].values

    logger.info("5) Jump & regime flags")
    jr = jump_and_regime(df)
    for c in jr.columns:
        df[c] = jr[c].values

    # tick_count placeholder to stay schema-compatible
    df["tick_count"] = df["tick_volume"].astype("float32")

    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    logger.info(f"Saved synthetic microstructure -> {out_path} | {len(df):,} rows")
    logger.info(
        "Null counts for key features:\n"
        f"{df[['tick_imbalance','ofi_window','spread_mean','vprof_poc_dist','kyle_lambda']].isna().sum().to_string()}"
    )
    return df


def parse_args():
    p = argparse.ArgumentParser(description="Build synthetic microstructure features from M1 OHLCV (no ticks required).")
    p.add_argument("--m1", required=True, help="Input M1 CSV (Dukascopy format)")
    p.add_argument("--out", required=True, help="Output enriched CSV path")
    p.add_argument("--vp-window", type=int, default=240, help="Rolling window for volume profile (bars)")
    p.add_argument("--bin-size", type=float, default=0.10, help="Price bin size in USD")
    p.add_argument("--ofi-window", type=int, default=20, help="Window for rolling tick imbalance / OFI proxy")
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    build_synthetic_microstructure(
        m1_path=args.m1,
        out_path=args.out,
        vp_window=args.vp_window,
        bin_size=args.bin_size,
        ofi_window=args.ofi_window,
    )



### Create triple_barrier_labels.py

In [ ]:
%%writefile triple_barrier_labels.py
#!/usr/bin/env python3
"""
triple_barrier_labels.py
------------------------
Replace the current fixed-horizon {BUY, HOLD, SELL} labels with
López de Prado's Triple-Barrier Method + Meta-Labeling.

Why it beats your current labeling
----------------------------------
Your `build_labels()` in train_ensemble_gpu.py uses:

    label = 1 if ret(h=5) > 0.0005 else (-1 if ret < -0.0005 else 0)

Problems:
  * Fixed horizon ignores the path — you label a +10 pip outcome even if
    you would have stopped out at -30 pips first.
  * Symmetric thresholds ignore realized volatility (ATR). In calm markets
    nothing is actionable; in news spikes, everything is a 1.
  * Same threshold for XAUUSD across all regimes biases the model.

Triple-Barrier fixes all three by simulating a *real* trade: an upper
barrier (take-profit), a lower barrier (stop-loss), and a vertical
barrier (timeout). The label is the *first* barrier hit.

Cost model (v2)
---------------
All barriers are computed from the *deteriorated* entry price (ask for
longs, bid for shorts) and all exit checks compare against the
*deteriorated* exit price (bid for long TP/SL, ask for short TP/SL).

This means:
  * TP requires MORE momentum to hit (correct — harder to win)
  * SL requires LESS of an adverse move to hit (correct — easier to lose)
  * Commission is expressed as synthetic spread widening so the math is
    internally consistent and not just a patch on the raw barriers.

Cost constants
--------------
  COMMISSION_PIPS = 0.00006   FTMO raw account: $6/lot round-trip on GBPUSD
                               ($3/lot/side * 2 sides = $6 = 0.6 pips)
  DEFAULT_SPREAD  = 0.00008   0.8 pip fallback when cs_spread column absent

Spread column resolution order
-------------------------------
  cs_spread  ->  spread_mean  ->  spread_est  ->  DEFAULT_SPREAD constant

Usage
-----
    # 1. Make triple-barrier labels
    python train_pipeline/triple_barrier_labels.py \\
        --data train_pipeline/data/xauusd_m1_synmicro.csv \\
        --out  train_pipeline/data/xauusd_m1_tb.csv \\
        --pt-atr 1.5 \\
        --sl-atr 1.0 \\
        --max-hold 30

    # 2. Train primary model on labels as before
    python train_pipeline/train_ensemble_gpu.py \\
        --data train_pipeline/data/xauusd_m1_tb.csv \\
        --label-col tb_label ...

    # 3. Train meta filter (after primary predictions are available)
    python train_pipeline/triple_barrier_labels.py --meta \\
        --data  train_pipeline/data/xauusd_m1_tb.csv \\
        --preds train_pipeline/reports/primary_preds.csv \\
        --out-model train_pipeline/models_gpu/meta_filter.txt
"""

from __future__ import annotations

import argparse
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("TripleBarrier")


# ---------------------------------------------------------------------------
# Cost model constants
# ---------------------------------------------------------------------------

# FTMO Raw account: $3/lot/side commission on GBPUSD
# $10/pip/lot => $6 round-trip = 0.6 pips = 0.00006 in price terms
COMMISSION_PIPS: float = 0.00006

# Fallback spread when the cs_spread / spread_mean / spread_est column is
# absent from the input CSV (e.g. raw MT5 data not yet through Step 1)
DEFAULT_SPREAD: float = 0.00008   # 0.8 pips


def _resolve_spread_series(df: pd.DataFrame) -> pd.Series:
    """Return the per-bar spread series from the DataFrame.

    Resolution order:
        cs_spread  ->  spread_mean  ->  spread_est  ->  DEFAULT_SPREAD constant

    The returned series is aligned to df.index.
    """
    for col in ("cs_spread", "spread_mean", "spread_est"):
        if col in df.columns:
            logger.info(f"[CostModel] Using spread column '{col}'")
            return df[col].astype("float64")
    logger.warning(
        f"[CostModel] No spread column found — falling back to "
        f"DEFAULT_SPREAD = {DEFAULT_SPREAD} ({DEFAULT_SPREAD * 1e4:.1f} pips). "
        "Run synthetic_microstructure.py first for per-bar spread estimates."
    )
    return pd.Series(DEFAULT_SPREAD, index=df.index, dtype="float64")


# ---------------------------------------------------------------------------
# Triple-barrier labeler
# ---------------------------------------------------------------------------

def triple_barrier_labels(
    df: pd.DataFrame,
    pt_atr: float = 1.5,
    sl_atr: float = 1.0,
    max_hold: int = 30,
    side_col: str | None = None,
) -> pd.DataFrame:
    """Compute triple-barrier outcomes for each bar.

    Barriers (cost-adjusted, v2)
    ----------------------------
    All barrier levels and hit-checks account for the bid/ask spread and
    round-trip commission.  Entry is at the worse price (ask for longs,
    bid for shorts); exit checks compare against the worse exit price
    (bid for longs, ask for shorts).

    effective_spread = cs_spread[i] + COMMISSION_PIPS
    half_spread      = effective_spread / 2

    Long:
        entry_ask = close[i] + half_spread
        upper     = entry_ask + pt_atr * ATR[i]   (TP — further away)
        lower     = entry_ask - sl_atr * ATR[i]   (SL — closer to bid)
        TP hit when  high[j] - half_spread >= upper
        SL hit when  low[j]  - half_spread <= lower

    Short:
        entry_bid = close[i] - half_spread
        upper     = entry_bid - pt_atr * ATR[i]   (TP at lower price)
        lower     = entry_bid + sl_atr * ATR[i]   (SL at higher price)
        TP hit when  low[j]  + half_spread <= upper
        SL hit when  high[j] + half_spread >= lower

    If `side_col` is provided (primary model's +/-1 signals), barriers are
    mirrored for shorts and the label becomes the meta-labeling target:
        +1  trade profitable (TP barrier hit first)
         0  vertical barrier expired
        -1  adverse barrier hit first

    If `side_col` is None, a long-side default is used and the three-class
    label (-1/0/+1) matches your existing convention but is now path-aware,
    volatility-scaled, and cost-adjusted.
    """
    required = {"close", "high", "low", "ATR"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    close = df["close"].values.astype("float64")
    high  = df["high"].values.astype("float64")
    low   = df["low"].values.astype("float64")
    atr   = df["ATR"].values.astype("float64")
    n     = len(df)

    spread_arr = _resolve_spread_series(df).values.astype("float64")

    side = (
        df[side_col].values.astype("int8")
        if side_col
        else np.ones(n, dtype="int8")
    )

    tb_label = np.zeros(n, dtype="int8")
    tb_ret   = np.zeros(n, dtype="float32")
    tb_hit   = np.full(n, -1, dtype="int32")

    for i in range(n):
        s = side[i]
        if s == 0:
            continue
        atr_i = atr[i]
        if np.isnan(atr_i) or atr_i <= 0:
            continue

        # --- Cost model ---------------------------------------------------
        effective_spread = spread_arr[i] + COMMISSION_PIPS
        half_spread      = effective_spread / 2.0

        if s > 0:
            # Long: enter at ask, exit (TP and SL) at bid
            entry    = close[i] + half_spread
            upper    = entry + pt_atr * atr_i
            lower    = entry - sl_atr * atr_i
        else:
            # Short: enter at bid, exit (TP and SL) at ask
            entry    = close[i] - half_spread
            upper    = entry - pt_atr * atr_i   # TP: price moves down
            lower    = entry + sl_atr * atr_i   # SL: price moves up

        # --- Barrier scan -------------------------------------------------
        end     = min(i + 1 + max_hold, n)
        hit     = -1
        outcome = 0

        for j in range(i + 1, end):
            hi_j, lo_j = high[j], low[j]

            if s > 0:
                # Long exit checks — compare against bid
                high_bid = hi_j - half_spread
                low_bid  = lo_j - half_spread
                if high_bid >= upper:
                    outcome = 1
                    hit = j
                    break
                if low_bid <= lower:
                    outcome = -1
                    hit = j
                    break
            else:
                # Short exit checks — compare against ask
                high_ask = hi_j + half_spread
                low_ask  = lo_j + half_spread
                if low_ask <= upper:    # TP: exit at ask, price moved down
                    outcome = 1
                    hit = j
                    break
                if high_ask >= lower:   # SL: exit at ask, price moved up
                    outcome = -1
                    hit = j
                    break

        if hit == -1:
            # Vertical barrier: use sign of terminal return in signal direction
            j       = end - 1
            ret     = (close[j] - entry) / entry * (1 if s > 0 else -1)
            outcome = 0
            hit     = j
        else:
            ret = (close[hit] - entry) / entry * (1 if s > 0 else -1)

        tb_label[i] = outcome
        tb_ret[i]   = ret
        tb_hit[i]   = hit

    out = df.copy()
    out["tb_label"]   = tb_label
    out["tb_return"]  = tb_ret
    out["tb_hit_idx"] = tb_hit
    return out


# ---------------------------------------------------------------------------
# Meta-labeling: given primary side predictions, produce secondary labels
# ---------------------------------------------------------------------------

def meta_labels_from_primary(
    df: pd.DataFrame,
    primary_col: str = "primary_pred",
    pt_atr: float = 1.5,
    sl_atr: float = 1.0,
    max_hold: int = 30,
) -> pd.DataFrame:
    """Compute triple-barrier outcomes along the primary model's side.

    meta_label = 1 if trade hit TP, else 0 (SL or timeout)
    This becomes the target for a binary "should we act" filter model.
    """
    if primary_col not in df.columns:
        raise ValueError(f"Primary column '{primary_col}' not found")
    out = triple_barrier_labels(df, pt_atr, sl_atr, max_hold, side_col=primary_col)
    out["meta_label"]  = (out["tb_label"] == 1).astype("int8")
    out["has_signal"]  = (out[primary_col] != 0).astype("int8")
    return out


# ---------------------------------------------------------------------------
# Sample-weighting utilities (AFML §4): uniqueness + time-decay
# ---------------------------------------------------------------------------

def compute_uniqueness_weights(tb_hit_idx: np.ndarray, n: int) -> np.ndarray:
    """Weight each sample by 1 / number of concurrent labels.

    Two labels are concurrent if their [entry, first-touch] windows overlap.
    This prevents the classifier from over-weighting periods where many
    samples share the same outcome.
    """
    concurrency = np.zeros(n, dtype="int32")
    for i in range(n):
        end = tb_hit_idx[i]
        if end < 0:
            continue
        concurrency[i:end + 1] += 1
    w = np.zeros(n, dtype="float32")
    for i in range(n):
        end = tb_hit_idx[i]
        if end < 0:
            w[i] = 0
            continue
        seg  = concurrency[i:end + 1]
        w[i] = float(np.mean(1.0 / np.maximum(seg, 1)))
    return w


def time_decay_weights(n: int, decay: float = 0.5) -> np.ndarray:
    """Linear time decay: oldest sample weight = `decay`, newest = 1.0."""
    return np.linspace(decay, 1.0, n, dtype="float32")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data",     required=True, help="Input CSV with ATR column")
    p.add_argument("--out",      default=None,  help="Output CSV path")
    p.add_argument("--pt-atr",   type=float,    default=1.5)
    p.add_argument("--sl-atr",   type=float,    default=1.0)
    p.add_argument("--max-hold", type=int,       default=30)
    p.add_argument("--meta",     action="store_true",
                   help="Run meta-labeling (requires --preds column 'primary_pred')")
    p.add_argument("--preds",    default=None,
                   help="CSV with primary_pred column aligned to data")
    args = p.parse_args()

    df = pd.read_csv(args.data)
    df.columns = [c.lower().strip() for c in df.columns]
    if "atr" in df.columns and "ATR" not in df.columns:
        df.rename(columns={"atr": "ATR"}, inplace=True)
    if "ATR" not in df.columns:
        logger.info("ATR not found — computing from OHLC")
        hl = df["high"] - df["low"]
        hc = (df["high"] - df["close"].shift()).abs()
        lc = (df["low"]  - df["close"].shift()).abs()
        df["ATR"] = pd.concat([hl, hc, lc], axis=1).max(axis=1).rolling(14).mean()

    if args.meta:
        if not args.preds:
            sys.exit("--meta requires --preds")
        preds              = pd.read_csv(args.preds)
        df["primary_pred"] = preds["primary_pred"].astype("int8")
        out = meta_labels_from_primary(
            df, pt_atr=args.pt_atr, sl_atr=args.sl_atr, max_hold=args.max_hold
        )
    else:
        out = triple_barrier_labels(
            df, pt_atr=args.pt_atr, sl_atr=args.sl_atr, max_hold=args.max_hold
        )

    # Uniqueness & time-decay weights for downstream trainer
    w_u              = compute_uniqueness_weights(out["tb_hit_idx"].values, len(out))
    w_t              = time_decay_weights(len(out))
    out["sample_weight"] = (w_u * w_t).astype("float32")

    out_path = args.out or args.data.replace(".csv", "_tb.csv")
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(out_path, index=False)
    logger.info(f"Saved {len(out):,} rows with triple-barrier labels -> {out_path}")
    if "tb_label" in out.columns:
        vc = out["tb_label"].value_counts().to_dict()
        logger.info(f"Label distribution: {vc}")
        logger.info(
            f"[CostModel] COMMISSION_PIPS={COMMISSION_PIPS} "
            f"DEFAULT_SPREAD={DEFAULT_SPREAD}"
        )


if __name__ == "__main__":
    main()



### Create sota_signal_generator.py

In [ ]:
%%writefile sota_signal_generator.py
#!/usr/bin/env python3
"""
sota_signal_generator.py
------------------------
A state-of-the-art XAUUSD M1 signal generator that is realistic given the
data you have (Dukascopy OHLCV + synthetic microstructure, no live ticks).

Architecture
============
        +-----------------------------+
        |  raw M1 (Dukascopy)         |
        +--------------+--------------+
                       |
          synthetic_microstructure.py   <-- OFI/spread/VPOC/Kyle-λ
                       |
          triple_barrier_labels.py      <-- path-aware vol-scaled labels
                       |
     +-----------------+------------------+
     |                                    |
 PRIMARY MODEL                      SECONDARY MODEL
 (side prediction)                  (meta-label filter)
  PatchTST-lite  ─────── side ────►  LightGBM binary "is this TP?"
  (transformer on 60-bar            Trained on the primary's signals
   patches of features)             + microstructure context
     |
     └── LightGBM ensemble fallback (already in your repo)

Why PatchTST-lite
-----------------
PatchTST (Nie et al. 2023) splits time series into non-overlapping patches
and treats them as tokens. It outperforms Informer/Autoformer on long
horizons with ~10x fewer params and trains on a single GPU in minutes.
The *lite* variant here keeps:
  - Patch embedding
  - 3-layer encoder with multi-head self-attention
  - Channel-independent representation (each feature learned separately)
  - Direct 3-class head (SELL/HOLD/BUY) with focal loss for class imbalance

If `torch` isn't installed, the script will still run the LightGBM
ensemble path — matching your existing behavior but using the better
labels and features.

Usage
-----
    # End-to-end (everything):
    python train_pipeline/sota_signal_generator.py \
        --data train_pipeline/data/xauusd_m1_synmicro_tb.csv \
        --out-dir train_pipeline/models_sota \
        --patch-len 12 \
        --seq-len 120 \
        --epochs 20 \
        --gpu

    # Inference / forward-test:
    python train_pipeline/sota_signal_generator.py --predict \
        --data train_pipeline/data/latest_window.csv \
        --out-dir train_pipeline/models_sota
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("SOTA")


# ---------------------------------------------------------------------------
# Feature set
# ---------------------------------------------------------------------------

FEATURE_COLS = [
    # Price context
    "close", "open", "high", "low",
    # Classical indicators (mirror existing pipeline)
    "RSI", "MACD", "Signal_Line", "MACD_Hist",
    "VWAP", "close_minus_vwap", "ATR", "BB_width",
    # Synthetic microstructure from synthetic_microstructure.py
    "tick_imbalance", "bid_ask_vol_imbalance",
    "spread_mean", "spread_std",
    "ofi_window", "of_pressure_flag",
    "kyle_lambda", "amihud_illiq",
    "vprof_poc_dist", "vprof_in_value_area",
    "vprof_hvn_flag", "vprof_lvn_flag",
    # Regime
    "jump_flag", "signed_vol_z", "vol_regime",
]

LABEL_COL_DEFAULT = "tb_label"  # from triple_barrier_labels.py
SAMPLE_WEIGHT_COL = "sample_weight"

LABEL_MAP = {-1: 0, 0: 1, 1: 2}
LABEL_UNMAP = {0: -1, 1: 0, 2: 1}


# ---------------------------------------------------------------------------
# Data prep
# ---------------------------------------------------------------------------

@dataclass
class PrepConfig:
    seq_len: int = 120       # lookback window in bars
    patch_len: int = 12      # patch size (10 patches/window)
    horizon: int = 5         # only used when falling back to fixed-horizon labels


def _select_available(df: pd.DataFrame, cols: List[str]) -> List[str]:
    keep = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.warning(f"Missing features (will be skipped): {missing}")
    return keep


def build_windows(
    df: pd.DataFrame,
    feat_cols: List[str],
    label_col: str,
    seq_len: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return raw 2D data for lazy slicing in WindowDataset."""
    X_feat = df[feat_cols].astype("float32").values
    y_raw = df[label_col].astype("int8").values
    w = df[SAMPLE_WEIGHT_COL].astype("float32").values if SAMPLE_WEIGHT_COL in df.columns \
        else np.ones(len(df), dtype="float32")

    n = len(df)
    usable = n - seq_len
    if usable <= 0:
        raise ValueError(f"Not enough rows ({n}) for seq_len={seq_len}")

    # Labels and weights aligned to the END of each window
    y = np.array([LABEL_MAP[int(y_raw[i + seq_len - 1])] for i in range(usable)], dtype="int64")
    sw = np.array([w[i + seq_len - 1] for i in range(usable)], dtype="float32")

    return X_feat, y, sw


# ---------------------------------------------------------------------------
# PatchTST-lite model
# ---------------------------------------------------------------------------

if HAS_TORCH:
    class PatchTSTLite(nn.Module):
        """Channel-independent PatchTST-lite encoder with 3-class head."""

        def __init__(
            self,
            n_features: int,
            seq_len: int,
            patch_len: int = 12,
            d_model: int = 64,
            n_heads: int = 4,
            n_layers: int = 3,
            dropout: float = 0.1,
            n_classes: int = 3,
        ):
            super().__init__()
            assert seq_len % patch_len == 0, "seq_len must be divisible by patch_len"
            self.n_features = n_features
            self.seq_len = seq_len
            self.patch_len = patch_len
            self.n_patches = seq_len // patch_len
            self.d_model = d_model

            # Channel-independent patch embedding: each feature is its own channel
            self.patch_embed = nn.Linear(patch_len, d_model)
            self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_model * 4,
                dropout=dropout,
                batch_first=True,
                activation="gelu",
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
            # Mix channel representations
            self.head = nn.Sequential(
                nn.Linear(d_model * n_features, d_model * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 2, n_classes),
            )

        def forward(self, x: "torch.Tensor") -> "torch.Tensor":
            # x: (B, L, F)
            B, L, Fn = x.shape
            # reshape to (B*F, n_patches, patch_len)
            x = x.permute(0, 2, 1).reshape(B * Fn, self.n_patches, self.patch_len)
            z = self.patch_embed(x) + self.pos_embed     # (B*F, P, d)
            z = self.encoder(z)                          # (B*F, P, d)
            z = z.mean(dim=1)                            # (B*F, d) -- patch pool
            z = z.reshape(B, Fn * self.d_model)          # (B, F*d)
            return self.head(z)


    class FocalLoss(nn.Module):
        """Focal loss for class-imbalanced classification (Lin et al. 2017)."""

        def __init__(self, gamma: float = 2.0, alpha: "torch.Tensor|None" = None):
            super().__init__()
            self.gamma = gamma
            self.alpha = alpha

        def forward(self, logits, target, weight=None):
            logp = F.log_softmax(logits, dim=-1)
            p = logp.exp()
            logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
            p_t = p.gather(1, target.unsqueeze(1)).squeeze(1)
            loss = -((1 - p_t) ** self.gamma) * logp_t
            if self.alpha is not None:
                loss = loss * self.alpha.to(logits.device)[target]
            if weight is not None:
                loss = loss * weight
            return loss.mean()


    class WindowDataset(Dataset):
        def __init__(self, X_feat, y, w, seq_len, mu=None, sd=None):
            self.X_feat = torch.from_numpy(X_feat)
            self.y = torch.from_numpy(y)
            self.w = torch.from_numpy(w)
            self.seq_len = seq_len
            self.mu = torch.from_numpy(mu) if mu is not None else None
            self.sd = torch.from_numpy(sd) if sd is not None else None

        def __len__(self):
            return len(self.y)

        def __getitem__(self, i):
            x = self.X_feat[i : i + self.seq_len]
            if self.mu is not None and self.sd is not None:
                x = (x - self.mu) / self.sd
            return x, self.y[i], self.w[i]


# ---------------------------------------------------------------------------
# Training / evaluation
# ---------------------------------------------------------------------------

def train_primary(
    X_feat: np.ndarray, y: np.ndarray, w: np.ndarray,
    out_dir: str,
    seq_len: int,
    patch_len: int = 12,
    epochs: int = 20,
    batch_size: int = 128,
    lr: float = 3e-4,
    device: str = "cpu",
):
    if not HAS_TORCH:
        logger.error("torch not installed — cannot train PatchTST-lite. "
                     "Use train_ensemble_gpu.py as fallback.")
        return None

    n_usable = len(y)
    n_feat = X_feat.shape[1]
    
    # 1. Walk-forward split (70/30) on window indices
    cut = int(n_usable * 0.7)
    
    # 2. Compute normalization stats on TRAINING SLICE ONLY (safeguard against leakage)
    train_feat_slice = X_feat[: cut + seq_len]
    mu = train_feat_slice.mean(0, keepdims=True)
    sd = train_feat_slice.std(0, keepdims=True) + 1e-6
    logger.info(f"Normalizing with train-only stats: mu_max={mu.max():.4f} sd_avg={sd.mean():.4f}")

    y_tr, y_va = y[:cut], y[cut:]
    w_tr, w_va = w[:cut], w[cut:]

    cls_counts = np.bincount(y_tr, minlength=3)
    inv = 1.0 / np.maximum(cls_counts, 1)
    alpha = torch.tensor(inv / inv.sum() * 3, dtype=torch.float32)
    logger.info(f"Class counts: {cls_counts.tolist()}  alpha={alpha.tolist()}")

    model = PatchTSTLite(n_features=n_feat, seq_len=seq_len,
                         patch_len=patch_len).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = FocalLoss(gamma=2.0, alpha=alpha)

    train_ds = WindowDataset(X_feat[:cut + seq_len], y_tr, w_tr, seq_len, mu, sd)
    val_ds = WindowDataset(X_feat[cut:], y_va, w_va, seq_len, mu, sd)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, 
                              drop_last=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, 
                            num_workers=0, pin_memory=True)

    best_f1 = -1.0
    os.makedirs(out_dir, exist_ok=True)
    model_path = os.path.join(out_dir, "patchtst_primary.pt")

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot = 0.0
            for xb, yb, wb in train_loader:
                xb = xb.to(device); yb = yb.to(device); wb = wb.to(device)
                logits = model(xb)
                loss = loss_fn(logits, yb, weight=wb)
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                tot += loss.item() * xb.size(0)
            sched.step()
            tr_loss = tot / len(train_loader.dataset)

            # Val macro-F1
            model.eval()
            preds, trues = [], []
            with torch.no_grad():
                for xb, yb, _ in val_loader:
                    xb = xb.to(device)
                    p = model(xb).argmax(-1).cpu().numpy()
                    preds.append(p); trues.append(yb.numpy())
            preds = np.concatenate(preds); trues = np.concatenate(trues)
            # macro F1 manually
            f1s = []
            for c in range(3):
                tp = ((preds == c) & (trues == c)).sum()
                fp = ((preds == c) & (trues != c)).sum()
                fn = ((preds != c) & (trues == c)).sum()
                prec = tp / max(tp + fp, 1)
                rec = tp / max(tp + fn, 1)
                f1 = 2 * prec * rec / max(prec + rec, 1e-9)
                f1s.append(f1)
            f1 = float(np.mean(f1s))
            logger.info(f"epoch {ep:03d}  train_loss={tr_loss:.4f}  val_macroF1={f1:.4f}")

            if f1 > best_f1:
                best_f1 = f1
                torch.save({"state": model.state_dict(),
                            "n_features": n_feat,
                            "seq_len": seq_len,
                            "patch_len": patch_len,
                            "mu": mu,
                            "sd": sd}, model_path)
    except KeyboardInterrupt:
        logger.info("Primary training interrupted by user! Keeping best checkpoint and stopping early.")

    logger.info(f"Best val macro-F1: {best_f1:.4f}  saved -> {model_path}")
    return model_path


# ---------------------------------------------------------------------------
# Meta filter (LightGBM binary classifier)
# ---------------------------------------------------------------------------

def train_meta_filter(
    df: pd.DataFrame,
    primary_pred_col: str,
    out_dir: str,
) -> str | None:
    """Train binary meta-label filter on top of primary signals."""
    if not HAS_LGB:
        logger.warning("LightGBM unavailable — skipping meta filter.")
        return None
    if "meta_label" not in df.columns:
        logger.warning("meta_label column missing — run triple_barrier_labels.py --meta first.")
        return None
    mask = df[primary_pred_col] != 0
    d = df.loc[mask].copy()
    feats = [c for c in FEATURE_COLS if c in d.columns]
    X = d[feats].astype("float32").values
    y = d["meta_label"].astype("int8").values

    cut = int(len(d) * 0.7)
    train = lgb.Dataset(X[:cut], label=y[:cut])
    val = lgb.Dataset(X[cut:], label=y[cut:], reference=train)
    params = dict(
        objective="binary",
        metric=["auc", "binary_logloss"],
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=30,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        verbose=-1,
    )
    try:
        model = lgb.train(params, train, num_boost_round=2000, valid_sets=[val],
                          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    except KeyboardInterrupt:
        logger.info("Meta training interrupted! Will try to save the current booster.")
        
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "meta_filter.txt")
    model.save_model(path)
    logger.info(f"Saved meta filter -> {path}")
    return path


# ---------------------------------------------------------------------------
# Inference (primary + meta combined)
# ---------------------------------------------------------------------------

def predict_combined(
    df: pd.DataFrame,
    primary_path: str,
    meta_path: str | None,
    seq_len: int,
    patch_len: int,
    threshold_meta: float = 0.55,
    device: str = "cpu",
) -> pd.DataFrame:
    """Return DataFrame with 'primary_pred', 'meta_score', 'final_signal'."""
    ckpt = torch.load(primary_path, map_location=device)
    
    feats = _select_available(df, FEATURE_COLS)
    if len(feats) != ckpt["n_features"]:
        logger.warning(f"Feature count mismatch! Model expects {ckpt['n_features']} but found {len(feats)}. Check if indicators were added.")

    X_feat = df[feats].astype("float32").values
    n = len(df)
    if n < seq_len:
        raise ValueError(f"Need at least {seq_len} rows, got {n}")

    # Normalize using mean/std from checkpoint
    mu = ckpt.get("mu")
    sd = ckpt.get("sd")
    if mu is None:
        # Fallback for old checkpoints
        logger.warning("Checkpoint missing normalization stats. Re-computing from input (LEAKAGE RISK).")
        mu = X_feat.mean(0, keepdims=True)
        sd = X_feat.std(0, keepdims=True) + 1e-6

    # Labels and weights are not needed for inference, use dummies
    usable = n - seq_len + 1
    dummy_y = np.zeros(usable, dtype="int64")
    dummy_w = np.ones(usable, dtype="float32")
    
    predict_ds = WindowDataset(X_feat, dummy_y, dummy_w, seq_len, mu, sd)
    predict_loader = DataLoader(predict_ds, batch_size=256, shuffle=False, 
                                num_workers=0, pin_memory=True)

    ckpt_state = ckpt["state"]
    model = PatchTSTLite(n_features=ckpt["n_features"], seq_len=ckpt["seq_len"],
                         patch_len=ckpt["patch_len"]).to(device)
    model.load_state_dict(ckpt_state)
    model.eval()

    probs_list = []
    with torch.no_grad():
        for xb, _, _ in predict_loader:
            logits = model(xb.to(device))
            probs = F.softmax(logits, dim=-1).cpu().numpy()
            probs_list.append(probs)
    
    probs = np.concatenate(probs_list, axis=0)
    primary = np.argmax(probs, axis=1)
    primary_cls = np.array([LABEL_UNMAP[c] for c in primary], dtype="int8")

    out = pd.DataFrame(index=df.index[seq_len - 1 :])
    out["p_sell"], out["p_hold"], out["p_buy"] = probs[:, 0], probs[:, 1], probs[:, 2]
    out["primary_pred"] = primary_cls

    # Meta filter
    if meta_path and HAS_LGB:
        meta = lgb.Booster(model_file=meta_path)
        X_meta = df[feats].astype("float32").iloc[seq_len - 1 :].values
        out["meta_score"] = meta.predict(X_meta)
        out["final_signal"] = np.where(
            (out["primary_pred"] != 0) & (out["meta_score"] >= threshold_meta),
            out["primary_pred"], 0,
        ).astype("int8")
    else:
        out["meta_score"] = np.nan
        out["final_signal"] = out["primary_pred"]

    return out


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--out-dir", default="train_pipeline/models_sota")
    p.add_argument("--label-col", default=LABEL_COL_DEFAULT)
    p.add_argument("--seq-len", type=int, default=120)
    p.add_argument("--patch-len", type=int, default=12)
    p.add_argument("--epochs", type=int, default=20)
    p.add_argument("--batch-size", type=int, default=128)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--gpu", action="store_true")
    p.add_argument("--predict", action="store_true",
                   help="Run inference only (requires trained artifacts in --out-dir)")
    p.add_argument("--meta-threshold", type=float, default=0.55)
    args = p.parse_args()

    if args.gpu and HAS_TORCH:
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            try:
                import torch_directml
                if torch_directml.is_available():
                    device = torch_directml.device()
                else:
                    device = "cpu"
            except ImportError:
                device = "cpu"
    else:
        device = "cpu"
    logger.info(f"Device: {device}")

    df = pd.read_csv(args.data)
    df.columns = [c.lower().strip() if c != "ATR" else c for c in df.columns]
    
    # Handle NaNs from technical indicators
    df.ffill(inplace=True)
    df.bfill(inplace=True)
    
    # Normalize to consistent casing for ATR + indicator columns
    rename_map = {c: c for c in df.columns}
    df.rename(columns=rename_map, inplace=True)

    if args.predict:
        primary_path = os.path.join(args.out_dir, "patchtst_primary.pt")
        meta_path = os.path.join(args.out_dir, "meta_filter.txt")
        if not os.path.exists(meta_path):
            meta_path = None
        out = predict_combined(df, primary_path, meta_path,
                               seq_len=args.seq_len, patch_len=args.patch_len,
                               threshold_meta=args.meta_threshold, device=device)
        out_path = os.path.join(args.out_dir, "live_predictions.csv")
        out.to_csv(out_path, index=False)
        logger.info(f"Predictions -> {out_path}")
        return

    # ---- Train primary ----
    feats = _select_available(df, FEATURE_COLS)
    if args.label_col not in df.columns:
        sys.exit(f"Label column '{args.label_col}' not found. "
                 f"Run triple_barrier_labels.py first.")
    X, y, w = build_windows(df, feats, args.label_col, args.seq_len)
    logger.info(f"X {X.shape}  y {y.shape} (classes: {np.bincount(y).tolist()})")

    primary_path = train_primary(
        X, y, w,
        out_dir=args.out_dir,
        seq_len=args.seq_len,
        patch_len=args.patch_len,
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        device=device,
    )

    # ---- Emit primary predictions over full set for meta training ----
    if primary_path and HAS_TORCH:
        with torch.no_grad():
            model_ckpt = torch.load(primary_path, map_location=device)
            mu = model_ckpt.get("mu")
            sd = model_ckpt.get("sd")
            
            model = PatchTSTLite(n_features=model_ckpt["n_features"],
                                 seq_len=model_ckpt["seq_len"],
                                 patch_len=model_ckpt["patch_len"]).to(device)
            model.load_state_dict(model_ckpt["state"])
            model.eval()
            # Re-build windows via Dataset for alignment and RAM safety
            dummy_y = np.zeros(len(y), dtype="int64")
            dummy_w = np.ones(len(y), dtype="float32")
            preds_ds = WindowDataset(X, dummy_y, dummy_w, args.seq_len, mu, sd)
            preds_loader = DataLoader(preds_ds, batch_size=256, shuffle=False, 
                                      num_workers=0, pin_memory=True)
            
            preds_full = []
            for xb, _, _ in preds_loader:
                p = model(xb.to(device)).argmax(-1).cpu().numpy()
                preds_full.append(p)
            preds_full = np.concatenate(preds_full)
            primary_cls = np.array([LABEL_UNMAP[c] for c in preds_full], dtype="int8")
        # Align with df: window i produces a label at bar i + seq_len - 1
        aligned = np.zeros(len(df), dtype="int8")
        aligned[args.seq_len - 1 : args.seq_len - 1 + len(primary_cls)] = primary_cls
        df["primary_pred"] = aligned
        preds_csv = os.path.join(args.out_dir, "primary_preds.csv")
        df[["primary_pred"]].to_csv(preds_csv, index=False)
        logger.info(f"Primary predictions -> {preds_csv}")

    # ---- Train meta filter ----
    if "primary_pred" in df.columns:
        train_meta_filter(df, "primary_pred", args.out_dir)

    # ---- Save a tiny config file for downstream consumers ----
    cfg = dict(
        seq_len=args.seq_len,
        patch_len=args.patch_len,
        features=feats,
        label_col=args.label_col,
        meta_threshold=args.meta_threshold,
    )
    with open(os.path.join(args.out_dir, "sota_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)
    logger.info("Done.")


if __name__ == "__main__":
    main()



### Create train_sota_v2.py

In [ ]:
%%writefile train_sota_v2.py
#!/usr/bin/env python3
"""
train_sota_v2.py
----------------
Improved trainer that fixes the three root causes of your current
0.39 F1 + "confidence == 1" problem:

1. Replaces focal loss + aggressive class-weighting with
   **class-balanced cross-entropy + label smoothing (eps=0.1)**.
   Focal+inverse-frequency on a 3-class M1 problem is what's pushing
   your softmax to the corners. Label smoothing bounds target probs
   at (1 - eps + eps/K) ≈ 0.93, so the network literally cannot
   saturate to 1.0 anymore — confidence becomes meaningful.

2. **Temperature scaling calibration** (Guo et al. 2017, ICML).
   After training, we fit a single scalar T on a held-out val set
   so that softmax(logits / T) has low ECE. One parameter, zero
   risk of overfit, typical ECE drop from ~0.2 -> 0.03.

3. **Purged walk-forward CV** (Lopez de Prado AFML §7) + time-decay
   class weights. Replaces the naive 70/30 split that leaks labels
   across the triple-barrier horizon.

Additional quality upgrades:
  - Saves per-feature mu/sd into the checkpoint so live inference
    uses the same normalization as training (your current live_
    sota_trading.py already expects this).
  - Adds meta-feature flags (is_asian, is_london, is_ny, dow) so
    the model learns session-specific behavior.
  - Sequence mixup (alpha=0.1) for regularization.
  - Early stopping on val macro-F1 with best-epoch restore.
  - Writes a calibrated_config.json that live_sota_trading.py can
    read to get the decision threshold and temperature.

Usage
-----
    # Assumes xauusd_m1_synmicro_tb.csv exists (from SOTA pipeline)
    python train_pipeline/train_sota_v2.py \
        --data train_pipeline/data/xauusd_m1_synmicro_tb.csv \
        --out-dir train_pipeline/models_sota_v2 \
        --seq-len 120 --patch-len 12 \
        --epochs 40 --batch-size 256 --gpu
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
try:
    import torch_directml
    HAS_DIRECTML = True
except ImportError:
    HAS_DIRECTML = False

from sota_signal_generator import (  # noqa: E402
    PatchTSTLite, FEATURE_COLS, LABEL_MAP, LABEL_UNMAP,
    _select_available, SAMPLE_WEIGHT_COL,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("SOTAv2")


# ---------------------------------------------------------------------------
# Extra features (session + weekday) — cheap & high-signal for XAUUSD
# ---------------------------------------------------------------------------

SESSION_FEATURES = [
    "dow_sin", "dow_cos", "hour_sin", "hour_cos",
]

def add_session_features(df: pd.DataFrame) -> pd.DataFrame:
    t = pd.to_datetime(df["time"], utc=True, errors="coerce")
    h = t.dt.hour
    df["is_asian"]   = ((h >= 0) & (h < 8)).astype("int8")
    df["is_london"]  = ((h >= 8) & (h < 16)).astype("int8")
    df["is_ny"]      = ((h >= 13) & (h < 21)).astype("int8")
    df["is_overlap"] = ((h >= 13) & (h < 16)).astype("int8")
    df["dow_sin"]  = np.sin(2*np.pi*t.dt.dayofweek/7).astype("float32")
    df["dow_cos"]  = np.cos(2*np.pi*t.dt.dayofweek/7).astype("float32")
    df["hour_sin"] = np.sin(2*np.pi*h/24).astype("float32")
    df["hour_cos"] = np.cos(2*np.pi*h/24).astype("float32")
    return df


# ---------------------------------------------------------------------------
# Dataset with sequence mixup
# ---------------------------------------------------------------------------

class LazyWindowDS(Dataset):
    def __init__(self, X_feat, y_raw, w_raw, seq_len, indices, mu, sd):
        self.X_feat = X_feat          # (N, F), np.ndarray
        self.y_raw = y_raw            # (N,)
        self.w_raw = w_raw            # (N,)
        self.seq_len = seq_len
        self.indices = indices        # array of usable window starts
        self.mu = mu
        self.sd = sd

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.seq_len
        
        # Slice window
        x = self.X_feat[start:end]    # (seq_len, F)
        
        # Label and weight from the LAST bar in the window
        # (Consistent with Triple Barrier mapping)
        y = LABEL_MAP[int(self.y_raw[end - 1])]
        w = self.w_raw[end - 1]

        # Apply normalization using TRAIN stats
        x = (x - self.mu) / (self.sd + 1e-9)

        x = torch.from_numpy(x.astype("float32"))
        y = torch.tensor(y, dtype=torch.long)
        w = torch.tensor(w, dtype=torch.float32)
        return x, y, w

def mixup_batch(x, y, w, alpha=0.1):
    """Sequence-level mixup: x_mix = λx_i + (1-λ)x_j on the *input* only."""
    if alpha <= 0:
        return x, y, w
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, w


def purged_split(n, embargo_frac=0.01, val_frac=0.2):
    """Return (train_idx, val_idx) with a purged gap of `embargo_frac`*n
    between end of train and start of val. Protects against triple-barrier
    label leakage where label_i depends on bars up to i+max_hold.
    """
    val_start = int(n * (1 - val_frac))
    embargo = int(n * embargo_frac)
    train_end = max(1, val_start - embargo)
    tr = np.arange(0, train_end)
    va = np.arange(val_start, n)
    return tr, va


# ---------------------------------------------------------------------------
# Calibration (temperature scaling)
# ---------------------------------------------------------------------------

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_T = nn.Parameter(torch.zeros(1))

    @property
    def T(self):
        return self.log_T.exp()

    def forward(self, logits):
        return logits / self.T


def fit_temperature(logits: torch.Tensor, y: torch.Tensor,
                    max_iter=200, lr=0.01) -> float:
    """Fit a single-param temperature on validation logits."""
    scaler = TemperatureScaler().to(logits.device)
    opt = torch.optim.LBFGS([scaler.log_T], lr=lr, max_iter=max_iter)
    loss_fn = nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad()
        loss = loss_fn(scaler(logits), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(scaler.T.item())


def expected_calibration_error(probs: np.ndarray, y: np.ndarray, bins=15) -> float:
    conf = probs.max(-1)
    pred = probs.argmax(-1)
    correct = (pred == y).astype("float32")
    ece = 0.0
    edges = np.linspace(0, 1, bins+1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            continue
        ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


# ---------------------------------------------------------------------------
# Macro-F1 helper
# ---------------------------------------------------------------------------

def macro_f1(y_true, y_pred, n_cls=3):
    f1s = []
    for c in range(n_cls):
        tp = ((y_pred == c) & (y_true == c)).sum()
        fp = ((y_pred == c) & (y_true != c)).sum()
        fn = ((y_pred != c) & (y_true == c)).sum()
        prec = tp / max(tp+fp, 1)
        rec  = tp / max(tp+fn, 1)
        f1   = 2*prec*rec / max(prec+rec, 1e-9)
        f1s.append(f1)
    return float(np.mean(f1s)), [float(x) for x in f1s]


# ---------------------------------------------------------------------------
# Main training
# ---------------------------------------------------------------------------

def train(args):
    if args.gpu:
        if torch.cuda.is_available():
            device = "cuda"
        elif HAS_DIRECTML:
            device = torch_directml.device()
        else:
            device = "cpu"
    else:
        device = "cpu"
    logger.info(f"device={device}")

    df = pd.read_csv(args.data)
    df.columns = [c.lower() if c != "ATR" else c for c in df.columns]
    if "atr" in df.columns and "ATR" not in df.columns:
        df.rename(columns={"atr": "ATR"}, inplace=True)

    df = add_session_features(df)

    feats = _select_available(df, FEATURE_COLS + SESSION_FEATURES)
    logger.info(f"Using {len(feats)} features")

    if args.label_col not in df.columns:
        sys.exit(f"{args.label_col} column missing — run triple_barrier_labels.py first.")

    # Extract arrays for lazy windowing after dropping NaNs
    df = df.dropna(subset=feats + [args.label_col]).reset_index(drop=True)
    X_feat = df[feats].astype("float32").values
    
    # Critical Safety Check
    nan_count = np.isnan(X_feat).sum()
    inf_count = np.isinf(X_feat).sum()
    if nan_count > 0 or inf_count > 0:
        logger.warning(f"Data contains {nan_count} NaNs and {inf_count} Infs! Removing them...")
        X_feat = np.nan_to_num(X_feat, nan=0.0, posinf=0.0, neginf=0.0)
    
    y_raw = df[args.label_col].astype("int8").values
    w_raw = df[SAMPLE_WEIGHT_COL].astype("float32").values if SAMPLE_WEIGHT_COL in df.columns \
            else np.ones(len(df), dtype="float32")

    usable = len(df) - args.seq_len
    tr, va = purged_split(usable, embargo_frac=0.01, val_frac=0.2)
    
    # Compute normalization statistics on TRAIN period portion only
    # To include all data covered by training windows: [0, max(tr) + seq_len)
    max_train_end = int(tr.max() + args.seq_len)
    X_train_slice = X_feat[:max_train_end]
    mu = X_train_slice.mean(axis=0, keepdims=True)
    sd = X_train_slice.std(axis=0, keepdims=True) + 1e-6
    
    logger.info(f"Data: total={len(df)}, usable_windows={usable}, train_range={max_train_end}")
    logger.info(f"Normalization: mu.shape={mu.shape}, sd.shape={sd.shape}")

    # Loaders with LazyWindowDS
    train_ds = LazyWindowDS(X_feat, y_raw, w_raw, args.seq_len, indices=tr, mu=mu, sd=sd)
    val_ds   = LazyWindowDS(X_feat, y_raw, w_raw, args.seq_len, indices=va, mu=mu, sd=sd)
    
    train_ld = DataLoader(
        train_ds, batch_size=args.batch_size, shuffle=True, drop_last=True,
        num_workers=4, pin_memory=True
    )
    val_ld = DataLoader(
        val_ds, batch_size=args.batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )

    # Class-balanced weights (Cui et al. 2019): w_c = (1-β) / (1-β^n_c)
    # Extract training labels to compute frequencies
    y_tr = np.array([LABEL_MAP[int(y_raw[idx + args.seq_len - 1])] for idx in tr])
    beta = 0.9999
    cls_counts = np.bincount(y_tr, minlength=3).astype("float64")
    eff_num = 1.0 - np.power(beta, cls_counts)
    cb = (1.0 - beta) / np.maximum(eff_num, 1e-9)
    cb = cb / cb.sum() * 3.0
    class_weights = torch.tensor(cb, dtype=torch.float32, device=device)
    logger.info(f"class-balanced weights: {cb.tolist()}")

    # Model
    model = PatchTSTLite(
        n_features=len(feats),
        seq_len=args.seq_len,
        patch_len=args.patch_len,
        d_model=args.d_model,
        n_heads=args.n_heads,
        n_layers=args.n_layers,
        dropout=args.dropout,
    ).to(device)

    # Label-smoothed CE with class-balanced weights
    loss_fn = nn.CrossEntropyLoss(weight=class_weights,
                                  label_smoothing=args.label_smoothing,
                                  reduction="none")
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=args.lr, total_steps=args.epochs*len(train_ld),
        pct_start=0.1, anneal_strategy="cos",
    )

    best_f1 = -1.0
    best_state = None
    patience, bad = args.patience, 0
    Path(args.out_dir).mkdir(parents=True, exist_ok=True)
    model_path = os.path.join(args.out_dir, "patchtst_primary.pt")
    ckpt_path = os.path.join(args.out_dir, "checkpoint.pt")
    
    start_epoch = 1
    
    # RESUME Logic
    if args.resume and os.path.exists(ckpt_path):
        logger.info(f"Attempting to resume from {ckpt_path}...")
        res_ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(res_ckpt["state"])
        opt.load_state_dict(res_ckpt["optimizer_state"])
        if "scheduler_state" in res_ckpt:
            sched.load_state_dict(res_ckpt["scheduler_state"])
        
        start_epoch = res_ckpt["epoch"] + 1
        best_f1 = res_ckpt["best_f1"]
        best_state = res_ckpt["state"]
        logger.info(f"Resumed from epoch {res_ckpt['epoch']} (Best F1: {best_f1:.4f})")

    for ep in range(start_epoch, args.epochs + 1):
        model.train()
        tot = 0.0
        n_batch = 0
        for xb, yb, wb in train_ld:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
            if args.mixup > 0:
                xb, yb, wb = mixup_batch(xb, yb, wb, alpha=args.mixup)
            logits = model(xb)
            per_sample = loss_fn(logits, yb)
            loss = (per_sample * wb).mean()
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            tot += float(loss.item()); n_batch += 1

        # Validate
        model.eval()
        all_logits, all_y = [], []
        with torch.no_grad():
            for xb, yb, _ in val_ld:
                xb = xb.to(device)
                all_logits.append(model(xb).cpu()); all_y.append(yb)
        logits_val = torch.cat(all_logits); y_val = torch.cat(all_y)
        preds = logits_val.argmax(-1).numpy()
        f1, f1_per = macro_f1(y_val.numpy(), preds)
        probs = F.softmax(logits_val, dim=-1).numpy()
        ece_raw = expected_calibration_error(probs, y_val.numpy())
        mean_conf = float(probs.max(-1).mean())
        logger.info(f"epoch {ep:03d} loss={tot/n_batch:.4f} "
                    f"macroF1={f1:.4f} per_cls={f1_per} "
                    f"mean_conf={mean_conf:.3f} ECE={ece_raw:.3f}")

        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
            
            # Save durable checkpoint
            torch.save({
                "epoch": ep,
                "state": best_state,
                "optimizer_state": opt.state_dict(),
                "scheduler_state": sched.state_dict(),
                "best_f1": best_f1,
                "n_features": len(feats),
                "seq_len": args.seq_len,
                "patch_len": args.patch_len,
                "d_model": args.d_model,
                "n_heads": args.n_heads,
                "n_layers": args.n_layers,
                "mu": mu,
                "sd": sd,
                "features": feats,
            }, ckpt_path)
            logger.info(f"Checkpoint saved to {ckpt_path}")
        else:
            bad += 1
            if bad >= patience:
                logger.info(f"Early stop at epoch {ep} (best F1={best_f1:.4f})")
                break

    # Reload best weights, fit temperature, save
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        all_logits, all_y = [], []
        for xb, yb, _ in val_ld:
            xb = xb.to(device)
            all_logits.append(model(xb).cpu()); all_y.append(yb)
    logits_val = torch.cat(all_logits); y_val = torch.cat(all_y)

    T = fit_temperature(logits_val, y_val)
    probs_cal = F.softmax(logits_val / T, dim=-1).numpy()
    ece_cal = expected_calibration_error(probs_cal, y_val.numpy())
    preds_cal = probs_cal.argmax(-1)
    f1_cal, _ = macro_f1(y_val.numpy(), preds_cal)
    mean_conf_cal = float(probs_cal.max(-1).mean())
    logger.info(f"calibration: T={T:.3f} ECE {ece_raw:.3f} -> {ece_cal:.3f}  "
                f"mean_conf {mean_conf:.3f} -> {mean_conf_cal:.3f}  F1={f1_cal:.4f}")

    # Save artifacts
    torch.save({
        "state": model.state_dict(),
        "n_features": len(feats),
        "seq_len": args.seq_len,
        "patch_len": args.patch_len,
        "d_model": args.d_model,
        "n_heads": args.n_heads,
        "n_layers": args.n_layers,
        "mu": mu.astype("float32").squeeze(0),   # (F,)
        "sd": sd.astype("float32").squeeze(0),
        "features": feats,
        "temperature": T,
    }, model_path)

    with open(os.path.join(args.out_dir, "sota_config.json"), "w") as f:
        json.dump({
            "seq_len": args.seq_len,
            "patch_len": args.patch_len,
            "features": feats,
            "label_col": args.label_col,
            "temperature": T,
            "meta_threshold": args.meta_threshold,
            "best_val_macro_f1": best_f1,
            "calibrated_macro_f1": f1_cal,
            "ece_raw": ece_raw,
            "ece_calibrated": ece_cal,
            "label_smoothing": args.label_smoothing,
        }, f, indent=2)
    logger.info(f"Saved -> {model_path}")


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--out-dir", default="train_pipeline/models_sota_v2")
    p.add_argument("--label-col", default="tb_label")
    p.add_argument("--seq-len", type=int, default=120)
    p.add_argument("--patch-len", type=int, default=12)
    p.add_argument("--epochs", type=int, default=40)
    p.add_argument("--batch-size", type=int, default=256)
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--d-model", type=int, default=96)
    p.add_argument("--n-heads", type=int, default=4)
    p.add_argument("--n-layers", type=int, default=3)
    p.add_argument("--dropout", type=float, default=0.15)
    p.add_argument("--label-smoothing", type=float, default=0.1)
    p.add_argument("--mixup", type=float, default=0.1)
    p.add_argument("--patience", type=int, default=6)
    p.add_argument("--meta-threshold", type=float, default=0.55)
    p.add_argument("--gpu", action="store_true")
    p.add_argument("--resume", action="store_true", help="Resume from checkpoint.pt if exists")
    return p.parse_args()


if __name__ == "__main__":
    train(parse_args())



### Create prune_features.py

In [ ]:
%%writefile prune_features.py
#!/usr/bin/env python3
"""
prune_features.py
-----------------
SHAP-based feature importance diagnostic for the GBPUSD LightGBM and PyTorch models.
"""

from __future__ import annotations

import argparse
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except ImportError:
    pass

try:
    import torch
    try:
        import torch_directml
    except ImportError:
        pass
except ImportError:
    sys.exit("torch not installed. Run: pip install torch")

try:
    import shap
except ImportError:
    sys.exit("shap not installed. Run: pip install shap")

# ---------------------------------------------------------------------------
# Core analysis
# ---------------------------------------------------------------------------

def run_shap_analysis(
    model_path: str,
    data_path: str,
    num_samples: int = 10_000,
    prune_pct: float = 0.20,
    out_dir: str | None = None,
) -> pd.DataFrame:
    model_path = Path(model_path)
    if not model_path.exists():
        sys.exit(f"Model not found: {model_path}")
    print(f"Loading model: {model_path}")

    # Detect model type from file extension
    if model_path.suffix == ".pt":
        print("Detected PyTorch model (.pt) — using DeepExplainer")
        device = torch.device("cpu")  # Force CPU for SHAP
        ckpt = torch.load(model_path, map_location=device)
        
        # Import the model class
        script_dir = Path(__file__).resolve().parent
        if str(script_dir) not in sys.path:
            sys.path.insert(0, str(script_dir))
        try:
            from train_sota_v2 import PatchTSTLite
        except ImportError:
            try:
                from sota_signal_generator import PatchTSTLite
            except ImportError:
                sys.exit("Could not import PatchTSTLite from train_sota_v2.py or sota_signal_generator.py")
        
        # Instantiate model using the saved checkpoint config
        model = PatchTSTLite(
            n_features=ckpt["n_features"],
            seq_len=ckpt.get("seq_len", 60),
            patch_len=ckpt.get("patch_len", 12),
            d_model=ckpt.get("d_model", 64),
            n_heads=ckpt.get("n_heads", 4),
            n_layers=ckpt.get("n_layers", 2)
        )
        model.load_state_dict(ckpt["state"])
        model.to(device)
        model.eval()
        
        features = ckpt.get("features", None)
        seq_len = ckpt.get("seq_len", 60)
        is_pytorch = True
    else:
        print("Detected LightGBM model (.txt) — using TreeExplainer")
        model = lgb.Booster(model_file=str(model_path))
        features = model.feature_name()
        is_pytorch = False

    if features is None:
        sys.exit("Could not determine features from model.")
    print(f"Model has {len(features)} features")

    # --- Load validation data -----------------------------------------------
    data_path = Path(data_path)
    if not data_path.exists():
        sys.exit(f"Data not found: {data_path}")
    print(f"Loading data: {data_path}")
    df = pd.read_csv(data_path)
    df.columns = [c.lower() if c != "ATR" else c for c in df.columns]
    if "atr" in df.columns and "ATR" not in df.columns:
        df.rename(columns={"atr": "ATR"}, inplace=True)

    if is_pytorch:
        try:
            from train_sota_v2 import add_session_features
            df = add_session_features(df)
        except Exception as e:
            print(f"Warning: could not add session features: {e}")

    missing = [f for f in features if f not in df.columns]
    if missing:
        sys.exit(f"Missing features in CSV:\n{missing}")

    # --- SHAP values --------------------------------------------------------
    print("Computing SHAP values (this may take 1–2 minutes)...")
    
    if is_pytorch:
        # We need num_samples + seq_len rows to build the sliding windows
        # so the final output has num_samples sequences
        rows_needed = num_samples + seq_len
        X_full = df[features].tail(rows_needed).values
        actual_n = len(X_full) - seq_len
        if actual_n <= 0:
            sys.exit(f"Not enough data to build {seq_len} sliding windows.")
            
        print(f"Using {actual_n:,} validation sequences for SHAP analysis")
        
        # Build sliding windows
        def build_windows(data, length):
            return np.array([data[i:i+length] for i in range(len(data) - length + 1)])
            
        windows = build_windows(X_full, seq_len)
        
        # We need background data (e.g. 20 samples) and validation data
        # DeepExplainer scales linearly with background samples; 100 was still too slow for a Transformer.
        bg_samples = min(20, len(windows))
        # Use num_samples for validation, defaulting to 1000 if user doesn't specify otherwise
        val_samples = min(num_samples, len(windows))
        
        bg_samples = min(2, len(windows))
        val_samples = min(10, len(windows))
        
        bg_tensor = torch.tensor(windows[:bg_samples], dtype=torch.float32).to(device)
        val_tensor = torch.tensor(windows[:val_samples], dtype=torch.float32).to(device)
        
        explainer = shap.DeepExplainer(model, bg_tensor)
        # Disable additivity check because DeepLIFT hooks don't perfectly support LayerNorm/Attention ops
        shap_values = explainer.shap_values(val_tensor, check_additivity=False)
        
        # DeepExplainer returns a list of tensors for classification
        if isinstance(shap_values, list):
            shap_values = [v.detach().cpu().numpy() if torch.is_tensor(v) else v for v in shap_values]
        elif torch.is_tensor(shap_values):
            shap_values = shap_values.detach().cpu().numpy()
            
        # IMPORTANT: DeepExplainer returns varying tensor shapes for sequence classification models
        # (e.g. (batch, seq_len, features, classes), (batch, classes, seq_len, features), or lists of tensors).
        # We must reduce over all non-feature dimensions to get per-feature importances!
        if isinstance(shap_values, list):
            # If it's a list, it's [class0, class1, class2], each of shape (batch, seq_len, features)
            shap_values = np.stack(shap_values, axis=0)  # (classes, batch, seq_len, features)
            
        mean_abs_shap = np.abs(shap_values)
        
        # Find the dimension that corresponds to the number of features
        feat_dim = None
        # We search from the end backwards, because batch/classes might accidentally match len(features)
        for i in reversed(range(len(mean_abs_shap.shape))):
            if mean_abs_shap.shape[i] == len(features):
                feat_dim = i
                break
                
        if feat_dim is None:
            sys.exit(f"Could not find feature dimension in SHAP output. Shape: {mean_abs_shap.shape}, expected features: {len(features)}")
            
        # Mean/Sum over all other axes
        axes_to_reduce = tuple(i for i in range(len(mean_abs_shap.shape)) if i != feat_dim)
        mean_abs_shap = mean_abs_shap.mean(axis=axes_to_reduce)
        print(f"SHAP squashed to 1D feature array: {mean_abs_shap.shape}")

    else:
        X_val = df[features].tail(num_samples).reset_index(drop=True)
        print(f"Using {len(X_val):,} rows for SHAP analysis")
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_val)

        if isinstance(shap_values, list):
            mean_abs_shap = np.zeros(len(features))
            for class_shap in shap_values:
                mean_abs_shap += np.abs(class_shap).mean(axis=0)
            print(f"Multi-class 2D SHAP: summed across {len(shap_values)} classes")
        else:
            mean_abs_shap = np.abs(shap_values).mean(axis=0)
            print("Binary 2D SHAP: single class")

    # --- Build ranking ------------------------------------------------------
    importance_df = pd.DataFrame({
        "feature":          features,
        "shap_importance":  mean_abs_shap,
    }).sort_values("shap_importance", ascending=True).reset_index(drop=True)

    # --- Save CSV -----------------------------------------------------------
    save_dir = Path(out_dir) if out_dir else model_path.parent
    save_dir.mkdir(parents=True, exist_ok=True)
    csv_path = save_dir / "feature_shap_ranking.csv"
    importance_df.to_csv(csv_path, index=False)
    print(f"\nFull ranking saved: {csv_path}")

    # --- Print prune candidates ---------------------------------------------
    prune_count = max(1, int(len(features) * prune_pct))
    prune_df    = importance_df.head(prune_count)
    top_df      = importance_df.tail(10).sort_values("shap_importance", ascending=False)

    sep = "=" * 60
    print(f"\n{sep}")
    print(
        f"BOTTOM {prune_count} FEATURES TO PRUNE "
        f"(bottom {prune_pct*100:.0f}% by SHAP — pure noise candidates)"
    )
    print(sep)
    print(prune_df.to_string(index=False))

    print(f"\n{sep}")
    print("TOP 10 FEATURES (sanity check — these should stay)")
    print(sep)
    print(top_df.to_string(index=False))

    print(f"\n{sep}")
    print("NEXT STEP: copy the feature names above into train_sota_v2.py:")
    print(sep)
    print("DROP_FEATURES = [")
    for feat in prune_df["feature"].tolist():
        print(f'    "{feat}",')
    print("]")
    print(sep)

    return importance_df

# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main():
    p = argparse.ArgumentParser(
        description="SHAP feature importance analysis for GBPUSD models"
    )
    p.add_argument(
        "--model",
        default="train_pipeline/reports/gbpusd/lgb_model.txt",
        help="Path to trained model file (.txt or .pt)",
    )
    p.add_argument(
        "--data",
        default="train_pipeline/data/gbpusd_m1_tb.csv",
        help="Path to triple-barrier labeled CSV",
    )
    p.add_argument(
        "--samples",
        type=int,
        default=10_000,
        help="Number of recent rows/sequences to use as validation set (default: 10000)",
    )
    p.add_argument(
        "--prune-pct",
        type=float,
        default=0.20,
        help="Fraction of features to flag for pruning (default: 0.20 = bottom 20%)",
    )
    p.add_argument(
        "--out-dir",
        default=None,
        help="Directory to save feature_shap_ranking.csv (default: same dir as model)",
    )
    args = p.parse_args()

    run_shap_analysis(
        model_path=args.model,
        data_path=args.data,
        num_samples=args.samples,
        prune_pct=args.prune_pct,
        out_dir=args.out_dir,
    )

if __name__ == "__main__":
    main()



### Execute Pipeline

In [ ]:
# 1. Create data and reports directories
!mkdir -p data
!mkdir -p reports/usdjpy

# 2. Run Synthetic Microstructure
# Make sure you uploaded usdjpy_m1.csv to the data folder!
!python synthetic_microstructure.py \
    --m1 data/usdjpy_m1.csv \
    --out data/usdjpy_m1_synmicro.csv \
    --vp-window 90 \
    --bin-size 0.01 \
    --ofi-window 15

# 3. Triple Barrier Labels
!python triple_barrier_labels.py \
    --data data/usdjpy_m1_synmicro.csv \
    --out data/usdjpy_m1_tb.csv \
    --pt-atr 3.0 \
    --sl-atr 0.8 \
    --max-hold 10

# 4. Train Model
!python train_sota_v2.py \
    --data data/usdjpy_m1_tb.csv \
    --out-dir reports/usdjpy \
    --seq-len 60 \
    --patch-len 8 \
    --epochs 50 \
    --patience 10 \
    --gpu

# 5. Prune Features (SHAP)
!python prune_features.py --model reports/usdjpy/patchtst_primary.pt --data data/usdjpy_m1_tb.csv --samples 200



### Download Model

In [ ]:
from google.colab import files
files.download('reports/usdjpy/patchtst_primary.pt')
files.download('reports/usdjpy/sota_config.json')
